In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import NMF

df = pd.read_csv("ratings.csv")

train, test = train_test_split(df, test_size=0.2, random_state=42)

n_users = df.userId.nunique()
n_items = df.movieId.nunique()

user_mapper = {u:i for i,u in enumerate(df.userId.unique())}
item_mapper = {m:i for i,m in enumerate(df.movieId.unique())}

train["u"] = train.userId.map(user_mapper)
train["i"] = train.movieId.map(item_mapper)

test["u"] = test.userId.map(user_mapper)
test["i"] = test.movieId.map(item_mapper)

R = np.zeros((n_users, n_items))

for row in train.itertuples():
    R[row.u, row.i] = row.rating

model = NMF(n_components=20, init="random", random_state=42, max_iter=500)
U = model.fit_transform(R)
V = model.components_

R_hat = np.dot(U, V)

preds = []
actuals = []

for row in test.itertuples():
    preds.append(R_hat[row.u, row.i])
    actuals.append(row.rating)

rmse = np.sqrt(mean_squared_error(actuals, preds))
mae = mean_absolute_error(actuals, preds)

print(rmse)
print(mae)

def recommend(user_id, n=10):
    uid = user_mapper[user_id]
    scores = R_hat[uid]
    return np.argsort(scores)[::-1][:n]

3.083657030428986
2.86729152717693
